# BRCA internal val — held-out `rep1`, cohort_safe seeds 42 and 44

Xenium BRCA has two sections. `rep1` is fold 0 and is the only slide drawn. `rep2` is fold 1.

These two runs are `l12_links_1` with `INNER_VAL=cohort_safe`. Each selected `best_mlp_gpu.pt` is fold 4, which trained on both BRCA slides, so the maps use `hce_group_fold_0.pt`, the weights that held `rep1` out. Seed 43 has no weight and is not listed. This notebook only reads predictions.

```bash
CUDA_DEVICE=1 bash code/Internal_val/demo_uni2_axisfix_brca.sh predict-ivsafe
```


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
if not (REPO / "code").is_dir():
    REPO = Path("/nobackup2/users/lingyu/Python/Hist2Pheno")
sys.path.insert(0, str(REPO / "code" / "Internal_val"))

import internal_val as iv

PAN = REPO / "data" / "PanCancer"
DATASET = "BRCA"
SAVE_FIGURES = False
iv.FOLD = 0

ARM = "l12_links_1"
TAG = "new1066axisfix_ivsafe"
SEEDS = (42, 44)


def ivsafe_paths(seed):
    result = PAN / f"result_uni2_ablation_{ARM}_{TAG}_paired_v2_seed{seed}"
    # Fold 4 best_mlp trained on both BRCA slides. Fold 0 held rep1 out.
    ckpt = result / "cross_dataset_cv" / ARM / "hce_group_fold_0.pt"
    pred_root = PAN / f"pred_uni2_ablation_{ARM}_{TAG}_paired_v2_seed{seed}_brca_fold0" / "oof"
    return pred_root, ckpt


MODELS = [
    (f"{ARM} seed {seed}", "cohort_safe, fold-0 checkpoint (held out rep1)", *ivsafe_paths(seed))
    for seed in SEEDS
]


def use_model(name, pred_root, ckpt):
    iv.PRED_ROOT = pred_root
    iv.CKPT = ckpt
    iv.FOLD_PLAN = ckpt.parents[2] / "fold_plan.csv"
    iv.METRICS = pred_root / "metrics_per_section.csv"
    iv.FIG_DIR = pred_root / "internal_val_figures"
    return iv.viz_samples(DATASET)


name0, note0, pred0, ckpt0 = MODELS[0]
viz0 = use_model(name0, pred0, ckpt0)
print(f"{DATASET} fold {iv.FOLD}: {len(MODELS)} models, drawing {viz0}")
for name, note, pred_root, ckpt in MODELS:
    pred = pred_root / "matched" / DATASET / "rep1" / "rep1_matched_pred.h5ad"
    print(f"{'ok ' if pred.is_file() else 'MISSING'}  {name}  {note}")


In [ ]:
for name, note, pred_root, ckpt in MODELS:
    viz = use_model(name, pred_root, ckpt)
    display(Markdown(f"## {name} \u2014 {note}"))
    missing = [s for s in viz if not iv.pred_paths(DATASET, s)["pred"].is_file()]
    if missing:
        print("missing", ", ".join(missing), "\u2014 CUDA_DEVICE=1 bash code/Internal_val/demo_uni2_axisfix_brca.sh predict-ivsafe")
        continue
    met = iv.metrics_fold0(DATASET)
    if not met.empty and "macro_auc_L2_cells" in met.columns:
        show = met.loc[met["sample"].astype(str).isin(viz), ["sample", "n_matched", "macro_auc_L2_cells"]]
        display(show)
    for sample in viz:
        fig = iv.plot_spatial_gt_pred(DATASET, sample, "L2", save=SAVE_FIGURES, title=f"{name}  {sample}")
        display(fig)
        plt.close(fig)
        fig = iv.plot_composition(DATASET, sample, "L2", save=SAVE_FIGURES, title=f"{name}  {sample}")
        display(fig)
        plt.close(fig)



In [ ]:
# GT and composition for every model are drawn in the cell above.

